# 09 - Inference Pipeline

This notebook provides the reusable inference pipeline for a new candidate CV and a selected job position.

The pipeline uses the same resources and scoring logic as the final matching system:

1. Text cleaning
2. TF-IDF similarity
3. Word2Vec similarity
4. Skill matching
5. Saved percentile scaling
6. Final score combination

The final score uses the project weights:
- 25% TF-IDF
- 25% Word2Vec
- 50% Skill Coverage

The `predict_match()` function is the single source of truth for inference so that the notebook, API, and application use the same scoring logic.

In [1]:
import os
import sys
import ast
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

PYTHON_FILES_DIR = ".."

if PYTHON_FILES_DIR not in sys.path:
    sys.path.insert(0, PYTHON_FILES_DIR)

from project_utils import (
    clean_text,
    tokenize_text,
    document_vector,
    build_skill_index,
    find_skills_in_text,
    skill_match_score,
    scale_value,
    combine_scores
)

In [2]:
DATA_PATH = os.path.join(
    "..",
    "Data",
    "final_matching_results.csv"
)

MODELS_DIR = os.path.join(
    "..",
    "Models"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Available columns:", list(df.columns))

Dataset shape: (9544, 46)
Available columns: ['career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'skills_required', 'matched_score', 'candidate_text', 'job_text', 'candidate_text_length', 'job_text_length', 'split', 'tfidf_similarity', 'absolute_error', 'word2vec_similarity', 'skill_coverage_percentage', 'skill_coverage_score', 'matched_skills', 'missing_skills', 'candidate_record_id', 'tfidf_scaled', 'word2vec_scaled', 'final_match_score', 'final_match_percentage']


In [3]:
tfidf_vectorizer = joblib.load(
    os.path.join(
        MODELS_DIR,
        "tfidf_vectorizer.pkl"
    )
)

word2vec_model = Word2Vec.load(
    os.path.join(
        MODELS_DIR,
        "word2vec_model.model"
    )
)

skill_resources = joblib.load(
    os.path.join(
        MODELS_DIR,
        "skill_resources.pkl"
    )
)

scaling = joblib.load(
    os.path.join(
        MODELS_DIR,
        "score_scaling.pkl"
    )
)

skill_index = build_skill_index(
    skill_resources["skill_vocabulary"]
)

skill_synonyms = skill_resources["skill_synonyms"]

print("All inference resources loaded successfully.")

All inference resources loaded successfully.


## Job Text Lookup

For a new CV, the selected job position is used to retrieve the corresponding job description from the final matching data.

In [4]:
def get_job_text(job_position):
    job_rows = df[
        df["job_position_name"]
        .astype(str)
        .str.lower()
        == str(job_position).lower()
    ].copy()

    if job_rows.empty:
        return None

    job_rows = job_rows[
        job_rows["job_text"].notna()
    ]

    if job_rows.empty:
        return None

    return clean_text(
        str(job_rows.iloc[0]["job_text"])
    )

## Single Inference Function

`predict_match()` performs the complete matching process.

The final score is calculated only here, preventing different cells from producing different final results.

In [5]:
def predict_match(candidate_text, job_position):
    job_text = get_job_text(job_position)

    if job_text is None:
        raise ValueError(
            "Job position was not found or has no job text."
        )

    candidate_text = clean_text(
        candidate_text
    )

    if not candidate_text.strip():
        raise ValueError(
            "No readable candidate text was found."
        )

    tfidf_score = cosine_similarity(
        tfidf_vectorizer.transform([candidate_text]),
        tfidf_vectorizer.transform([job_text])
    )[0][0]

    candidate_vector = document_vector(
        tokenize_text(candidate_text),
        word2vec_model
    )

    job_vector = document_vector(
        tokenize_text(job_text),
        word2vec_model
    )

    if (
        np.linalg.norm(candidate_vector) > 0
        and
        np.linalg.norm(job_vector) > 0
    ):
        word2vec_score = cosine_similarity(
            [candidate_vector],
            [job_vector]
        )[0][0]
    else:
        word2vec_score = 0.0

    candidate_skills = find_skills_in_text(
        candidate_text,
        skill_index,
        skill_synonyms
    )

    required_skills = find_skills_in_text(
        job_text,
        skill_index,
        skill_synonyms
    )

    coverage, matched_skills, missing_skills = skill_match_score(
        candidate_skills,
        required_skills
    )

    tfidf_scaled = scale_value(
        tfidf_score,
        **scaling["tfidf_similarity"]
    )

    word2vec_scaled = scale_value(
        word2vec_score,
        **scaling["word2vec_similarity"]
    )

    final_match_score = combine_scores(
        tfidf_scaled,
        word2vec_scaled,
        coverage
    )

    return {
        "job_position_name": job_position,
        "tfidf_similarity": float(tfidf_score),
        "word2vec_similarity": float(word2vec_score),
        "skill_coverage_score": (
            None
            if pd.isna(coverage)
            else float(coverage)
        ),
        "skill_coverage_percentage": (
            None
            if pd.isna(coverage)
            else float(coverage * 100)
        ),
        "tfidf_contribution": float(
            tfidf_scaled * 0.25
        ),
        "word2vec_contribution": float(
            word2vec_scaled * 0.25
        ),
        "skill_contribution": (
            None
            if pd.isna(coverage)
            else float(coverage * 0.50)
        ),
        "final_match_score": float(
            final_match_score
        ),
        "final_match_percentage": float(
            final_match_score * 100
        ),
        "matched_skills": list(
            matched_skills
        ),
        "missing_skills": list(
            missing_skills
        )
    }

## Test the Inference Pipeline

The following example uses an existing candidate and job pair from the dataset.

This is only a pipeline test. The function above is the reusable inference logic for new CVs.

In [6]:
sample_row = df.iloc[0]

sample_candidate_text = clean_text(
    str(sample_row["candidate_text"])
)

sample_job_position = str(
    sample_row["job_position_name"]
)

result = predict_match(
    sample_candidate_text,
    sample_job_position
)

pd.DataFrame([result])

,job_position_name,tfidf_similarity,word2vec_similarity,skill_coverage_score,skill_coverage_percentage,tfidf_contribution,word2vec_contribution,skill_contribution,final_match_score,final_match_percentage,matched_skills,missing_skills
0,senior software engineer,0.003533,0.692982,0.0,0.0,0.019825,0.168079,0.0,0.187904,18.790385,[],"[documentation, software engineer, technical s..."


## Consistency Check

For an existing candidate-job pair, the inference score should be close to the stored final matching score.

Small differences can occur if the stored dataset was generated with a different intermediate version, but the inference calculation itself must use one consistent scoring function.

In [7]:
if "final_match_percentage" in df.columns:
    stored_score = pd.to_numeric(
        sample_row["final_match_percentage"],
        errors="coerce"
    )

    predicted_score = result[
        "final_match_percentage"
    ]

    if pd.notna(stored_score):
        difference = abs(
            float(stored_score)
            - float(predicted_score)
        )

        print(
            f"Stored final score: {float(stored_score):.2f}%"
        )
        print(
            f"Inference final score: {float(predicted_score):.2f}%"
        )
        print(
            f"Difference: {difference:.4f} percentage points"
        )
    else:
        print(
            "The selected row has no stored final score."
        )
else:
    print(
        "final_match_percentage is not available in the dataset."
    )

Stored final score: 18.79%
Inference final score: 18.79%
Difference: 0.0000 percentage points


## Example Output

The inference result contains:

- TF-IDF similarity
- Word2Vec similarity
- Skill coverage
- Individual score contributions
- Final match percentage
- Matched skills
- Missing skills

The same `predict_match()` logic can be used by the application/API for new CV uploads.

The system is intended as a recruitment decision-support tool and should not be used as an automatic hiring decision.